# 08 · Mecanismos da virada de 2018 (série E)

**Testes confirmatórios.** Especificações registradas em `docs/hipoteses.md` ("Série E")
antes de abrir cada dado novo. O diagnóstico exploratório está no `07_virada_2018.ipynb`.

**Como cada mecanismo é testado.** Uma medida X (padronizada: o coeficiente é o efeito de
1 desvio-padrão) entra na regressão da variação do PT em cada transição entre 1ºs turnos,
comparando AMCs da mesma região imediata, com o mesmo tamanho, o mesmo nível de partida e
os mesmos controles estruturais (renda, alfabetização, % urbana, idade, cor ou raça). Um
mecanismo da virada precisa de uma **pegada no tempo**:

1. efeito em 2014→2018 com o sinal previsto (IC 95% sem o zero);
2. **salto** em relação a 2010→2014: a diferença entre os dois efeitos também exclui o zero.

Se só (1) passar, a medida marca uma **clivagem antiga**, não um choque de 2018. O veredito
final de cada teste principal (★) sai depois da correção de Holm sobre todos eles; até lá,
os resultados são **provisórios**.

| Hipótese | Medida | Situação |
|---|---|---|
| E2a ★ | % evangélica, Censo 2010 | **sustenta-se** |
| E2c ★ | voto em candidatos religiosos a deputado federal | **sustenta-se** |
| E2b ★, E7a ★ | igrejas e MEIs no CNPJ | aguardando dados |
| E1 ★, E3 ★, E4 ★, E5 ★, E7b ★ | emendas e prefeitos, 4G, emprego, homicídios, conta própria | aguardando dados |

In [ ]:
import warnings

import altair as alt
import polars as pl
import pyfixest as pf

from kingsrace import paths
from kingsrace.features.base import geografia_amc
from kingsrace.models.event_study import CLUSTER, CONTROLES, base_transicoes, por_transicao, salto
from kingsrace.viz import tema  # noqa: F401  (registra e ativa o tema do projeto)
from kingsrace.viz.tema import SERIES, TINTA

alt.renderers.enable("jupyter", offline=True)
warnings.filterwarnings("ignore", message=".*singleton fixed effect")

P = paths.PROCESSED
painel = pl.read_parquet(P / "painel_amc.parquet")
geo = geografia_amc(pl.read_parquet(P / "municipios.parquet"))
cov = pl.read_parquet(P / "covariaveis_amc.parquet")
zero = alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(y=alt.datum(0))

PAPEIS = ["placebo (antes)", "alvo (2014→2018)", "depois"]
PAPEL = {
    "2006→2010": PAPEIS[0],
    "2010→2014": PAPEIS[0],
    "2014→2018": PAPEIS[1],
    "2018→2022": PAPEIS[2],
    "2022→2026": PAPEIS[2],
}


def grafico_event_study(res, titulo):
    base = alt.Chart(res.with_columns(papel=pl.col("transicao").replace_strict(PAPEL))).encode(
        x=alt.X("transicao:N", title=None, sort=None, axis=alt.Axis(labelAngle=0)),
        color=alt.Color(
            "papel:N",
            sort=PAPEIS,
            scale=alt.Scale(domain=PAPEIS, range=[TINTA["apagada"], *SERIES[:2]]),
            title=None,
        ),
        tooltip=[
            alt.Tooltip("transicao:N", title="Transição"),
            alt.Tooltip("coef:Q", title="Efeito de 1 dp", format="+.3f"),
            alt.Tooltip("ic_inf:Q", title="IC inf.", format="+.3f"),
            alt.Tooltip("ic_sup:Q", title="IC sup.", format="+.3f"),
            alt.Tooltip("p:Q", title="p", format=".4f"),
            alt.Tooltip("amcs:Q", title="AMCs"),
        ],
    )
    return (
        zero
        + base.mark_rule(strokeWidth=2).encode(
            y=alt.Y("ic_inf:Q", title="Efeito de 1 dp de X sobre Δ logit do PT"), y2="ic_sup:Q"
        )
        + base.mark_circle(size=90, opacity=1).encode(y="coef:Q")
    ).properties(width=520, height=240, title=titulo)


def robustez(base):
    specs = {
        "principal": por_transicao(base),
        "sem controles": por_transicao(base, controles=False),
        "com pesos": por_transicao(base, pesos=True),
    }
    return (
        pl.concat([r.with_columns(spec=pl.lit(nome)) for nome, r in specs.items()])
        .pivot(on="transicao", index="spec", values="coef")
        .with_columns(pl.all().exclude("spec").round(3))
    )


def mostra_salto(s):
    print(
        f"β(2014→2018) = {s['beta_alvo']:+.4f}   β(2010→2014) = {s['beta_anterior']:+.4f}\n"
        f"salto = {s['diferenca']:+.4f}  (IC 95% {s['ic_inf']:+.4f} a {s['ic_sup']:+.4f}; "
        f"p = {s['p']:.4f})"
    )


testes = {}  # p-valores dos testes principais (★), para a correção de Holm no fim

## E2a · % evangélica no Censo 2010

**Registrado:** X = % evangélica no Censo 2010, anterior às duas eleições de 2014. Previsão:
β < 0 em 2014→2018 (mais evangélicos, mais afastamento do PT) e salto negativo em relação a
2010→2014. A C1 já tinha mostrado, **no nível**, o efeito evangélico aparecendo em 2018; aqui
o teste é na variação, dentro da região imediata e com o placebo.

In [ ]:
base_e2a = base_transicoes(painel, geo, cov, cov.select("amc_id", x=pl.col("pct_evangelica_2010")))
e2a = por_transicao(base_e2a)
s_e2a = salto(base_e2a)
mostra_salto(s_e2a)
e2a.with_columns(pl.col("coef", "ic_inf", "ic_sup").round(4), pl.col("p").round(4))

**Como ler:** cada ponto é o efeito de 1 desvio-padrão de X sobre a variação do PT naquela
transição, com IC de 95%. Abaixo de zero, onde X é maior o PT perdeu mais (ou ganhou menos).
Cinza: placebos (antes de 2014); azul: a transição-alvo; laranja: depois.

In [ ]:
grafico_event_study(e2a, "E2a · % evangélica (Censo 2010) e a variação do PT")

In [ ]:
robustez(base_e2a)

**E2a se sustenta.**

- **Placebos limpos:** −0,006 em 2006→2010 e +0,007 em 2010→2014, os dois com IC em torno
  de zero.
- **Alvo:** −0,091 em 2014→2018 (IC −0,109 a −0,073). Entre AMCs da mesma região imediata e
  com o mesmo perfil, 1 desvio-padrão a mais de evangélicos em 2010 (9,5 pp) significou 0,09
  a menos no logit do PT, ou ~2,3 pp numa AMC dividida ao meio.
- **Salto:** −0,098 (IC −0,122 a −0,075).
- **Depois:** o efeito continua em 2018→2022 (−0,036) e se inverte levemente em 2022→2026
  (+0,012), em linha com a pista da A1 (com o resto fixo, o PT caiu menos onde havia mais
  evangélicos).
- **Robustez:** −0,107 sem controles e −0,188 com pesos pelo eleitorado (o efeito é maior nas
  AMCs grandes). Os placebos ficam pequenos em todas as especificações (o maior é +0,041, em
  2010→2014 com pesos).

## E2c · Voto em candidatos religiosos a deputado federal

**Registrado:** X = % dos válidos para deputado federal dados a candidatos religiosos
(ocupação "sacerdote ou membro de ordem ou seita religiosa" ou título evangélico no nome de
urna, sem "Padre" e "Frei"), medida **na eleição inicial de cada transição** (2014 para
2014→2018; 2010 para o placebo 2010→2014). Previsão: β < 0 em 2014→2018 e salto negativo.
É uma medida da **organização política** das igrejas, não da composição religiosa. Regra e
particularidades em `docs/decisoes.md` (D6).

In [ ]:
religiosos = pl.read_parquet(P / "religiosos_amc.parquet")
print("Candidatos religiosos, % dos válidos para deputado federal no país:")
print(
    religiosos.group_by("ano")
    .agg(pct=(100 * pl.col("votos_religiosos").sum() / pl.col("validos_deputado").sum()).round(2))
    .sort("ano")
)
base_e2c = base_transicoes(
    painel, geo, cov, religiosos.select("amc_id", de="ano", x="pct_religiosos")
)
e2c = por_transicao(base_e2c)
s_e2c = salto(base_e2c)
mostra_salto(s_e2c)
e2c.with_columns(pl.col("coef", "ic_inf", "ic_sup").round(4), pl.col("p").round(4))

In [ ]:
grafico_event_study(e2c, "E2c · Voto em candidatos religiosos e a variação do PT")

In [ ]:
robustez(base_e2c)

**E2c se sustenta.**

- **Placebos:** +0,001 e −0,000.
- **Alvo:** −0,027 em 2014→2018 (IC −0,040 a −0,014). 1 desvio-padrão (2,3 pp dos válidos
  para deputado indo a candidatos religiosos) equivale a ~0,7 pp do PT numa AMC dividida ao
  meio. É menor que a E2a.
- **Salto:** −0,027 (IC −0,044 a −0,010; p = 0,002).
- **Depois:** quase some (−0,006 em 2018→2022, IC tocando o zero; 0,000 em 2022→2026). A
  pegada da organização política das igrejas está concentrada em 2018.
- **Robustez:** −0,034 sem controles e −0,039 com pesos.
- **Nota:** a fatia nacional dos candidatos religiosos teve o pico em 2014 (1,88%) e caiu
  depois. A medida pega só a presença religiosa explícita (título ou ocupação) e é um limite
  inferior.

## Exploratório · composição ou organização?

A E2a mede **quantos** evangélicos há; a E2c, quanto as igrejas **se organizam na política**
(votos em pastores e bispos). As duas andam juntas: a pergunta é se a organização acrescenta
algo além da composição. Mesmo modelo de 2014→2018, com as duas medidas juntas.
**Exploratório**, sem registro: não entra no veredito.

In [ ]:
juntas = (
    base_e2a.filter(pl.col("transicao") == "2014→2018")
    .rename({"x": "evangelica_2010"})
    .join(
        base_e2c.filter(pl.col("transicao") == "2014→2018").select("amc_id", religiosos_2014="x"),
        on="amc_id",
    )
)
print(
    "correlação entre as duas medidas:",
    round(juntas.select(pl.corr("evangelica_2010", "religiosos_2014")).item(), 3),
)
fit = pf.feols(
    "dl ~ evangelica_2010 + religiosos_2014 + log_aptos + nivel + "
    + " + ".join(CONTROLES)
    + " | cd_regiao_imediata",
    juntas.to_pandas(),
    vcov=CLUSTER,
)
fit.tidy().loc[["evangelica_2010", "religiosos_2014"], ["Estimate", "2.5%", "97.5%"]].round(4)

As duas medidas quase não se correlacionam (0,18), e as duas mantêm o efeito quando entram
juntas: −0,090 (composição) e −0,019 (organização). Onde as igrejas já lançavam pastores e
bispos com voto em 2014, o PT perdeu mais em 2018, **além** do que a proporção de
evangélicos explicaria. É a primeira pista de que a **organização política** das igrejas, e
não só a religião dos eleitores, deixou pegada na virada. A E2b (igrejas no CNPJ) testa outra
face dessa organização.

## Testes principais até aqui (provisório)

Cada teste principal tem duas condições (efeito em 2014→2018 e salto) que precisam valer
juntas. O p-valor do teste é o **maior** dos dois (teste de interseção-união: a conclusão
conjunta só é tão forte quanto a mais fraca). A correção de Holm é aplicada quando todos os
★ estiverem calculados.

In [ ]:
for nome, res, s in [("E2a", e2a, s_e2a), ("E2c", e2c, s_e2c)]:
    p_alvo = res.filter(pl.col("transicao") == "2014→2018")["p"].item()
    testes[nome] = max(p_alvo, s["p"])
pl.DataFrame({"teste": list(testes), "p (interseção-união)": list(testes.values())})

## Até aqui

- **As duas medidas de igreja têm a pegada de um mecanismo da virada:** nada antes de 2014 e
  um salto em 2014→2018.
- **Os vereditos já são definitivos.** Com 11 testes principais (★), um p ≤ 0,05/11 ≈ 0,0045
  garante a rejeição em Holm, quaisquer que sejam os outros resultados. E2a
  (p ≈ 3 × 10⁻¹⁵) e E2c (p = 0,002) estão abaixo disso.
- **O que isto não diz:**
  - Não é prova causal. Qualquer coisa que tenha "ligado" em 2018 e que se concentre onde há
    mais evangélicos produz a mesma pegada. A candidatura de Bolsonaro e sua pauta de
    costumes, ou a campanha de desinformação dirigida a evangélicos no WhatsApp, fazem parte
    do canal "igreja" no sentido amplo, mas não são a mesma coisa que uma mobilização
    organizada pelas igrejas. A E3 (internet móvel) e a E6 (todas juntas) ajudam a separar.
  - Também não diz **quem** organizou ou financiou essa mobilização. Isso é trabalho da
    linha documental.
- **Próximo:** E2b e E7a (CNPJ: igrejas e MEIs), depois E4/E5, E3 e E1.